In [ ]:
# for plotting
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import colorcet as cc
import numpy as np
import pickle
import json
from pathlib import Path

sns.set()
sns.set_context('poster')
sns.set_style('ticks')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = 'cmr10'
plt.rcParams["mathtext.fontset"] = 'cm'
plt.rcParams["axes.formatter.use_mathtext"] = True

In [ ]:
import h5py

In [ ]:
def calculate_csd(sim_path, scratch_path=None):
    """Calculates cloud statistics for a given simulation.

    Parameters
    ----------
    sim_path    : path containing uninterrupted_large_clouds.json and pkl/qclm.pkl
    scratch_path: path containing hdf5/*.h5 (defaults to sim_path if not given)

    Memory layout:
      Phase 1: HDF5 scanned in CHUNK-row slices — peak ~1.5 GB (was ~30 GB for full load).
      Phase 2/3: plume arrays loaded for attached clouds only.
    Vectorized mcbl_c/mctl_c and mass-flux averaging (no Python loops).
    """

    dx, dy, dz = 250, 250, 50   # m
    grid_vol   = dx * 1e-3 * dy * 1e-3 * dz * 1e-3  # km³
    cbl_gap    = 3
    qclm_crit  = 1.0e-6
    CHUNK      = 5000   # rows per HDF5 read; ~780 MB transient per iteration

    sim_path     = Path(sim_path)
    scratch_path = Path(scratch_path) if scratch_path is not None else sim_path

    with open(sim_path / 'uninterrupted_large_clouds.json') as f:
        ul_clouds = json.load(f)
    nc_json = len(ul_clouds)

    # Consistency check: NC must agree across all three HDF5 files and the JSON
    hdf5_files = {
        'cloud_all_af.h5': 'af',
        'plume_all_af.h5': 'af',
        'plume_all_mf.h5': 'mf',
    }
    nc_shapes = {}
    for fname, ds_key in hdf5_files.items():
        with h5py.File(scratch_path / 'hdf5' / fname) as f:
            nc_shapes[fname] = f[ds_key].shape[0]

    if len(set(nc_shapes.values())) > 1:
        raise ValueError(
            f"NC mismatch across HDF5 files:\n" +
            "\n".join(f"  {k}: NC={v}" for k, v in nc_shapes.items())
        )
    nc_hdf5 = next(iter(nc_shapes.values()))
    if nc_json != nc_hdf5:
        raise ValueError(
            f"NC mismatch: uninterrupted_large_clouds.json has {nc_json} clouds "
            f"but HDF5 files have {nc_hdf5} rows"
        )

    # Phase 1: chunked scan — never holds the full (NC, NT, NZ) array in memory
    with h5py.File(scratch_path / 'hdf5/cloud_all_af.h5') as f:
        ds = f['af']
        nc, nt, nz = ds.shape

        cloud_volumes  = np.zeros((nc, nt), dtype=np.float32)
        area_z         = np.zeros((nc, nz), dtype=np.float32)
        max_cloud_area = np.zeros(nc,       dtype=np.float32)
        cbl_c          = np.zeros((nc, nt), dtype=np.int16)

        for i0 in range(0, nc, CHUNK):
            i1    = min(i0 + CHUNK, nc)
            chunk = ds[i0:i1]                              # (cs, NT, NZ) float32
            cloud_volumes[i0:i1]  = chunk.sum(axis=2)
            area_z[i0:i1]         = chunk.sum(axis=1)
            max_cloud_area[i0:i1] = chunk.max(axis=(1, 2))
            cbl_c[i0:i1]          = np.argmax(chunk > 0, axis=2).astype(np.int16)

    # Derived quantities from the small accumulated arrays
    cloud_times         = (cloud_volumes > 0).sum(axis=1)
    cloud_ini_time      = np.argmax(cloud_volumes > 0, axis=1)
    total_cloud_volumes = cloud_volumes.sum(axis=1) * grid_vol

    has_area = area_z > 0
    mcbl_c   = np.argmax(has_area, axis=1)
    mctl_c   = (nz - 1) - np.argmax(has_area[:, ::-1], axis=1)
    del area_z, has_area

    with open(sim_path / 'pkl/qclm.pkl', 'rb') as f:
        qclm = np.asarray(pickle.load(f))
    mcbl = np.argmax(qclm > qclm_crit, axis=1)   # (NT,)

    attached_c   = (cloud_volumes > 0) & (cbl_c.astype(np.int32) - cbl_gap < mcbl[np.newaxis, :])
    del cbl_c
    attached_ind   = np.where(attached_c.any(axis=1))[0]
    attached_c_att = attached_c[attached_ind, :]
    del attached_c, cloud_volumes

    # Phase 2: plume area — attached rows only
    with h5py.File(scratch_path / 'hdf5/plume_all_af.h5') as f:
        max_plume_area = f['af'][attached_ind].max(axis=(1, 2))

    # Phase 3: plume mass flux — attached rows only, vectorized
    with h5py.File(scratch_path / 'hdf5/plume_all_mf.h5') as f:
        plume_mfs_att = f['mf'][attached_ind].astype(np.float64) * (dx * dy)

    t_idx = np.arange(nt)
    l_idx = np.clip(mcbl - 1, 0, nz - 3)
    mf_t  = (
        plume_mfs_att[:, t_idx, l_idx    ] +
        plume_mfs_att[:, t_idx, l_idx + 1] +
        plume_mfs_att[:, t_idx, l_idx + 2]
    ) / 3.0
    del plume_mfs_att

    attached_sum = attached_c_att.sum(axis=1).astype(np.float64)
    mean_cb_mf   = np.where(
        attached_sum > 0,
        (attached_c_att * mf_t).sum(axis=1) / np.maximum(attached_sum, 1),
        0.0,
    )
    del mf_t, attached_c_att

    mean_cb_mf_c = np.where(mean_cb_mf > 1.0, mean_cb_mf, 1.1)

    return (
        mean_cb_mf_c, mean_cb_mf,
        total_cloud_volumes[attached_ind], cloud_times[attached_ind],
        mcbl_c[attached_ind], mctl_c[attached_ind],
        cloud_ini_time[attached_ind], max_cloud_area[attached_ind],
        max_plume_area, attached_ind,
    )

In [ ]:
ctl = 'goamazon_2pulse.largedom_1024.r20260116'
ehe1 ='goamazon_2pulse.largedom_1024.ehe1.r20260115'

In [ ]:
l_calc_csd = True
if l_calc_csd:
    stat_ctl = calculate_csd(f'{ctl}/', f'{ctl}/scratch/')
    with open(f'{ctl}/pkl/csd_stats.sparse.claude.pkl', 'wb') as f:
        pickle.dump(stat_ctl, f)
    stat_ehe1 = calculate_csd(f'{ehe1}/', f'{ehe1}/scratch/')
    with open(f'{ehe1}/pkl/csd_stats.sparse.claude.pkl', 'wb') as f:
        pickle.dump(stat_ehe1, f)

In [ ]:
with open(f'{ctl}/pkl/csd_stats.sparse.claude.pkl', 'rb') as f:
    stat_ctl = pickle.load(f)
with open(f'{ehe1}/pkl/csd_stats.sparse.claude.pkl', 'rb') as f:
    stat_ehe1 = pickle.load(f)

In [ ]:
nc_ehe1, = stat_ehe1[0].shape
nc_ctl, = stat_ctl[0].shape
print(f'{nc_ehe1} clouds in EHE1 simulation')
print(f'{nc_ctl} clouds in CTL simulation')